# Package Qwen/Qwen3-8B for offline Kaggle runs

Generated by `scripts/build_package_8b_notebooks.py`. Edit the generator, not this file.

Run with **Internet ON** (CPU is enough). It writes `/kaggle/working/models/Qwen3-8B/` and `MODELS.json`.
Offline RTX Pro 6000 kernels attach this kernel's output with `kernel_sources`.

In [ ]:
import json
import os
from datetime import datetime

from huggingface_hub import HfApi, snapshot_download

REPO = 'Qwen/Qwen3-8B'
OUT = '/kaggle/working/models'
LIMIT_GB = 20  # /kaggle/working size limit
ALLOW = ['*.json', '*.safetensors', 'merges.txt', '*.md', '*.txt']
IGNORE = ['onnx/*', 'openvino/*', 'original/*', '*.gguf', '*.bin', '*.onnx', '*.pt', '*.msgpack', '*.h5']
os.makedirs(OUT, exist_ok=True)

info = HfApi().model_info(REPO, files_metadata=True)
print(f'{REPO}: sha {info.sha}, created {info.created_at}, last modified {info.last_modified}')
name = REPO.split('/')[-1]
target = os.path.join(OUT, name)
snapshot_download(repo_id=REPO, revision=info.sha, local_dir=target, allow_patterns=ALLOW, ignore_patterns=IGNORE)
files = []
for root, dirs, names in os.walk(target):
    dirs[:] = [d for d in dirs if d != '.cache']  # local_dir metadata, not model files
    for file_name in names:
        if file_name.startswith('.'):
            continue
        path = os.path.join(root, file_name)
        files.append(dict(path=os.path.relpath(path, target), bytes=os.path.getsize(path)))
size_gb = sum(f['bytes'] for f in files) / 1e9
manifest = {REPO: dict(folder=f'models/{name}', sha=info.sha, created_at=str(info.created_at),
                       last_modified=str(info.last_modified), size_bytes=sum(f['bytes'] for f in files),
                       files=sorted(files, key=lambda f: f['path']))}
manifest_doc = dict(downloaded_at=datetime.now().isoformat(timespec='seconds'), total_gb=round(size_gb, 2),
                    limit_gb=LIMIT_GB, models=manifest,
                    rule='open models <= 15B released before 2026-08-01 (OVERVIEW.md)')
with open(os.path.join(OUT, 'MODELS.json'), 'w', encoding='utf-8') as file:
    json.dump(manifest_doc, file, ensure_ascii=False, indent=2)
print(f'{REPO}: sha {info.sha}, {size_gb:.2f} GB, {len(files)} files of /kaggle/working limit {LIMIT_GB} GB')
assert size_gb < LIMIT_GB, 'output exceeds the /kaggle/working limit'

In [ ]:
names = sorted(os.listdir(target))
print(name, names)
with open(os.path.join(target, 'config.json'), encoding='utf-8') as file:
    config = json.load(file)
print('architectures', config.get('architectures'), '| hidden_size', config.get('hidden_size'),
      '| layers', config.get('num_hidden_layers'), '| max_position_embeddings', config.get('max_position_embeddings'))
assert config.get('hidden_size') == 4096 and any('Qwen3' in a for a in config.get('architectures', []))
shards = [n for n in names if n.endswith('.safetensors')]
assert shards and 'tokenizer.json' in names
index = os.path.join(target, 'model.safetensors.index.json')
if os.path.exists(index):  # every shard named in the index must be present
    with open(index, encoding='utf-8') as file:
        wanted = set(json.load(file)['weight_map'].values())
    assert wanted <= set(names), f'missing shards: {sorted(wanted - set(names))}'